# DeliveryOps Intelligence System — Modelling

## Objective
Building machine learning models to:
- Predict delivery time (Regression)
- Predict delay risk (Classification)

We will also export models for deployment in a Streamlit app.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import joblib

## 1) Load cleaned dataset

We load the cleaned dataset

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_delivery.csv")
df.head()

,order_id,agent_age,agent_rating,store_latitude,store_longitude,drop_latitude,drop_longitude,order_date,order_time,pickup_time,...,vehicle,area,delivery_time,category,delay_flag,distance_km,order_hour,day_of_week,is_weekend,pickup_delay
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,...,motorcycle,Urban,120,Clothing,0,3.025149,11,5,1,15.0
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,...,scooter,Metropolitian,165,Electronics,1,20.183530,19,4,0,5.0
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,...,motorcycle,Urban,130,Sports,1,1.552758,8,5,1,15.0
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,...,motorcycle,Metropolitian,105,Cosmetics,0,7.790401,18,1,0,10.0
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,...,scooter,Metropolitian,150,Toys,1,6.210138,13,5,1,15.0


## 2. Validate Dataset
Ensure required engineered features exist.

## 3. Define Features and Targets

In [3]:
feature_cols = [
    "distance_km",
    "agent_rating",
    "order_hour",
    "day_of_week",
    "pickup_delay",
    "traffic",
    "weather",
    "vehicle",
    "area",
    "category"
]
target_reg = "delivery_time"
target_cls = "delay_flag"

X = df[feature_cols].copy()
y_reg = df[target_reg].copy()
y_cls = df[target_cls].copy()

X.head()

,distance_km,agent_rating,order_hour,day_of_week,pickup_delay,traffic,weather,vehicle,area,category
0,3.025149,4.9,11,5,15.0,High,Sunny,motorcycle,Urban,Clothing
1,20.183530,4.5,19,4,5.0,Jam,Stormy,scooter,Metropolitian,Electronics
2,1.552758,4.4,8,5,15.0,Low,Sandstorms,motorcycle,Urban,Sports
3,7.790401,4.7,18,1,10.0,Medium,Sunny,motorcycle,Metropolitian,Cosmetics
4,6.210138,4.6,13,5,15.0,High,Cloudy,scooter,Metropolitian,Toys


## 4. Preprocessing Pipeline
- Encode categorical variables
- Keep numeric features as-is

In [4]:
cat_cols = ["traffic", "weather", "vehicle", "area", "category"]
num_cols = ["distance_km", "agent_rating", "order_hour", "day_of_week", "pickup_delay"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

## 5. Regression Model — Predict Delivery Time
Model: RandomForestRegressor  
Metric: MAE (Mean Absolute Error)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_model = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(random_state=42))
])

reg_model.fit(X_train, y_train)

pred = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mae

### Result
MAE represents the average prediction error in minutes.
Lower values indicate better model performance.

## 6. Classification Model — Predict Delay Risk

Model: RandomForestClassifier  
Metrics:
- Accuracy
- ROC-AUC

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

clf_model = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(random_state=42))
])

clf_model.fit(X_train, y_train)

proba = clf_model.predict_proba(X_test)[:, 1]
pred_cls = (proba >= 0.5).astype(int)

acc = accuracy_score(y_test, pred_cls)
auc = roc_auc_score(y_test, proba)

acc, auc

### Result
- Accuracy shows overall correctness
- ROC-AUC shows how well the model distinguishes delayed vs non-delayed deliveries

## 7. Feature Importance (Model Interpretation)
Understanding what drives delivery time.

In [ ]:
# Extract trained model
model = reg_model.named_steps["model"]

# Get feature names after encoding
feature_names = reg_model.named_steps["prep"].get_feature_names_out()

importances = model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp.head(10)

### Insight
Top features influencing delivery time include distance, traffic conditions, and pickup delay, confirming their operational importance.

## 8. Save Models for Deployment

In [ ]:
joblib.dump(reg_model, "../app/delivery_time_model.joblib")
joblib.dump(clf_model, "../app/delay_risk_model.joblib")

print("✅ Models saved to app folder")

## 9. Conclusion

This project demonstrates how data analytics and machine learning can improve delivery operations by:

- Predicting delivery time for better planning  
- Identifying high-risk deliveries  
- Supporting operational decision-making  

The models can be deployed via a Streamlit app for real-time use.